# Fix a mosaic-shift-induced FOV gap

A worked example of diagnosing and patching a real acquisition-planning bug,
kept as a template for the same class of problem on other samples --
`notebooks/tests/` holds notebooks like this one, built to investigate/fix a
specific real issue rather than run as a standing part of the numbered
`prepare_imaging` pipeline.

**The bug this fixes**: the usual workflow is (1) scan a low-mag (10x) Steve
mosaic of the whole coverslip, (2) image a handful of FOVs at the real
imaging objective ("60x" here) over part of that scan to measure the fixed
stage-calibration offset between the two objectives, (3) shift the 10x
mosaic's stage positions by that offset so it lines up with real (60x)
coordinates, THEN (4) derive the tissue boundary and FOV grid from the
shifted mosaic. If step 3 is skipped (or the shift is computed but applied
too late, after the boundary was already saved), the boundary -- and every
FOV grid built from it -- ends up offset from where the tissue actually is.

**Sections**:
1. Load the raw Steve mosaic tiles (cached locally -- a slow, many-small-file
   read over a network drive) and composite BOTH objectives together
   (unshifted) into one image -- real tissue content, not just tile-center
   coordinates, so a real misalignment is visible right away.
2. Same composite, but with the known `(SHIFT_DX_UM, SHIFT_DY_UM)` correction
   applied to the low-mag tiles first -- the high-mag patch should now look
   like a seamless part of the surrounding tissue.
3. Assemble the shifted mosaic (low-mag only, real un-normalized values) and
   overlay the CURRENT (already-imaging) positions file's own FOV perimeters
   on top of it, to see the real-world misalignment directly.
4. Re-run tissue segmentation on the shifted mosaic (same parameters as this
   sample's own local `02_create_boundary_from_mosaic.ipynb`), then find FOV
   positions in that new boundary using the SAME dense grid the current
   positions file's own boundary was filtered from -- not an independently
   re-centered grid -- so any FOV the two boundaries share ends up at the
   EXACT same coordinate, not just an approximately-overlapping one.
5. Overlay the OLD vs. NEW tissue boundary outlines.
6. Classify every NEW FOV as already-covered (an exact coordinate match in
   the OLD grid) or MISSING, and count/report the missing ones.
7. Append the missing FOVs -- re-ordered into their own short-travel loop --
   to the end of the current positions array and save a new
   `positions_{tag}_added.txt`, ready to be imaged in a follow-up loop.

Does not touch the original positions file -- only ever writes a new,
distinctly-named one, and every plot is saved to `SAMPLE_DIR/figures/` for
visual review before trusting the result.

## 1 — Setup

In [ ]:
import os
import sys
import pickle
import dataclasses
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from shapely.geometry import box
from shapely.affinity import translate

MERCI_DIR  = Path(os.getcwd()).parent.parent   # MERci/ (notebook lives in MERci/notebooks/tests/)
SAMPLE_DIR = MERCI_DIR.parent                  # experiment root
sys.path.insert(0, str(MERCI_DIR / "src"))

from MERci.common.experiment_info import resolve_sample_identity, positions_file_tag
from MERci.common.io              import save_positions_array, read_image_frames
from MERci.acquisition.configs    import (
    get_fov_geometry, find_frame_table_for_hal_config, get_color_frame_indices,
)
from MERci.acquisition.mosaic     import (
    load_steve_mosaic, assemble_mosaic_canvas, segment_mosaic_tissue, plot_mosaic_segmentation,
)
from MERci.acquisition.positions  import (
    load_boundary_polygon, load_hole_polygons, create_grid_positions,
    generate_scanning_path, filter_scanning_path, get_path_stats,
)
from MERci.progress_display       import ProgressReporter

print(f"SAMPLE_DIR: {SAMPLE_DIR}")

## 2 — Parameters

In [ ]:
SAMPLE_NAME, IMAGING_DIR = resolve_sample_identity(MERCI_DIR)
POSITIONS_TAG = positions_file_tag(SAMPLE_NAME, IMAGING_DIR)
POSITIONS_DIR = SAMPLE_DIR / "positions"
FIGURES_DIR   = SAMPLE_DIR / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
NOTEBOOK_NAME = "fix_mosaic_shift_missing_fovs"
CACHE_DIR     = SAMPLE_DIR / "analysis" / "cache" / NOTEBOOK_NAME
CACHE_DIR.mkdir(parents=True, exist_ok=True)

# ── Mosaic source ──────────────────────────────────────────────────────────
MOSAIC_DIR  = SAMPLE_DIR / "data" / "mosaic10x"
MOSAIC_NAME = None   # None = auto-detect the only *.msc file present

# ── The known 10x -> real-imaging-objective calibration shift ─────────────
# Externally measured (comparing a handful of real-imaging-objective FOVs
# against the same tissue features in the 10x scan) -- NOT computed by this
# notebook, just applied. Added to every LOW-MAG tile's own (x_um, y_um).
LOW_MAG_OBJECTIVE  = "10x"
HIGH_MAG_OBJECTIVE = "60x"
SHIFT_DX_UM = 410.0
SHIFT_DY_UM = 420.0

# ── Segmentation parameters -- copied verbatim from this sample's own local
# 02_create_boundary_from_mosaic.ipynb (MERci/notebooks/prepare_imaging/
# lineage_tracing/merfish/), NOT re-derived, so the corrected boundary is
# produced the same way the original (uncorrected) one was. ──────────────
MOSAIC_KEEP_OBJECTIVES = [LOW_MAG_OBJECTIVE]   # exclude the high-mag calibration tiles from segmentation
WORKING_PIXEL_UM       = 5.0
THRESHOLD              = 400
SMOOTH_SIGMA_UM        = 10.0
CLOSE_RADIUS_UM        = 50.0
OPEN_RADIUS_UM         = 15.0
MARGIN_UM              = 25.0
MIN_TISSUE_AREA_UM2    = 4_000_000.0
MIN_HOLE_AREA_UM2      = 12_000.0
MIN_ISLAND_AREA_UM2    = 50_000.0
SIMPLIFY_TOL_UM        = 15.0

# ── FOV-grid parameters -- copied verbatim from this sample's own local
# 02_create_positions_from_boundaries.ipynb. ──────────────────────────────
MICROSCOPE           = "ST2"
non_overlap_fraction = 0.9
SCAN_DIRECTION       = "vertical"
pixel_size_um, image_size_px = get_fov_geometry(MICROSCOPE)
step_size_um = pixel_size_um * image_size_px * non_overlap_fraction
fov_size_um  = pixel_size_um * image_size_px

# ── Display normalization for the composited both-objectives views (steps
# 1-2 below) -- the two objectives are shot at very different exposures, so
# a single shared percentile-based stretch over the composited canvas lets
# the high-mag patch's saturated values wash out any real visual comparison.
# Each objective's own tiles are independently rescaled onto a common
# display range before compositing instead: low-mag keeps its own natural
# contrast range, high-mag is stretched from its own (narrower) range onto
# the SAME target scale. Display-only -- the real segmentation/grid steps
# below (4 onward) use the low-mag tiles at their own real, un-rescaled
# pixel values, exactly like the local 02_create_boundary_from_mosaic.ipynb.
LOW_MAG_DISPLAY_PCT  = (1.0, 99.0)
HIGH_MAG_DISPLAY_PCT = (2.0, 95.0)

CURRENT_POSITIONS_PATH = POSITIONS_DIR / f"positions_{POSITIONS_TAG}.txt"
ADDED_POSITIONS_PATH   = POSITIONS_DIR / f"positions_{POSITIONS_TAG}_added.txt"

# ── Real per-FOV image access -- for the empty-vs-has-cells analysis near
# the end of this notebook (section 13). Deliberately NOT
# ExperimentConfig/ExperimentMetadata: round_info.csv's own `dir` column can
# hold an absolute path from a DIFFERENT machine (this sample's is a stale
# `U:\...` path) -- ExperimentMetadata eagerly resolves every FOV of every
# round at load time, and confirmed directly that when that first-choice
# candidate is unreachable (not just "doesn't exist", but a slow OS-level
# auth failure), the combined per-candidate cost across 1000+ FOVs x 14
# rounds makes plain construction impractically slow, even after fixing the
# separate crash that same case caused (see common/metadata.py's new
# `_path_exists_safe`). Reading `round_info.csv` directly for just the cells
# round's own `series` pattern, and resolving raw files relative to THIS
# machine's own (confirmed-reachable) `SAMPLE_DIR`, sidesteps the slow
# candidate entirely.
IMAGE_SUFFIX = ".zarr"   # must match what HAL wrote

print(f"Sample name        : {SAMPLE_NAME}")
print(f"step_size_um       : {step_size_um:.1f}")
print(f"fov_size_um        : {fov_size_um:.1f}")
print(f"Shift (~{SHIFT_DX_UM/step_size_um:.2f} x {SHIFT_DY_UM/step_size_um:.2f} FOVs): "
      f"dX={SHIFT_DX_UM}, dY={SHIFT_DY_UM} um")
print(f"Current positions  : {CURRENT_POSITIONS_PATH}")
print(f"Will write         : {ADDED_POSITIONS_PATH}")

## 3 — Load the raw Steve mosaic (cached -- slow over a network drive)

Reading every tile's own `.stv` pickle is many small file opens over
(typically) a network-mounted acquisition drive -- slow regardless of total
data size (dominated by per-file latency, not throughput). Cached to
`analysis/cache/fix_mosaic_shift_missing_fovs/steve_tiles.pkl`, invalidated
by the `.msc` manifest's own mtime, per `NOTEBOOK_GUIDELINES.md` #2/#3.

In [ ]:
msc_candidates = sorted(MOSAIC_DIR.glob(f"{MOSAIC_NAME or '*'}.msc"))
if not msc_candidates:
    raise FileNotFoundError(f"No .msc mosaic manifest found in {MOSAIC_DIR}.")
if len(msc_candidates) > 1:
    print(f"WARNING: {len(msc_candidates)} .msc files found, using the first: {msc_candidates[0].name}.")
MSC_PATH = msc_candidates[0]

tiles_cache = CACHE_DIR / "steve_tiles.pkl"
msc_mtime   = MSC_PATH.stat().st_mtime

cached_ok = False
if tiles_cache.exists():
    with open(tiles_cache, "rb") as fh:
        cached = pickle.load(fh)
    cached_ok = cached.get("msc_mtime") == msc_mtime
    if cached_ok:
        tiles_all = cached["tiles"]
        print(f"Loaded {len(tiles_all)} cached tile(s): {tiles_cache}")

if not cached_ok:
    print(f"Reading {MSC_PATH} (every tile's own .stv file -- can take a couple of minutes "
          f"over a network drive, cached afterward)...")
    tiles_all = load_steve_mosaic(MSC_PATH)
    with open(tiles_cache, "wb") as fh:
        pickle.dump({"msc_mtime": msc_mtime, "tiles": tiles_all}, fh)
    print(f"Loaded and cached {len(tiles_all)} tile(s): {tiles_cache}")

obj_counts = Counter(t.objective_name for t in tiles_all)
print(f"Objective breakdown: {dict(obj_counts)}")
for obj in (LOW_MAG_OBJECTIVE, HIGH_MAG_OBJECTIVE):
    if obj not in obj_counts:
        raise ValueError(f"No tiles found for objective={obj!r} -- check LOW_MAG_OBJECTIVE/"
                          f"HIGH_MAG_OBJECTIVE against the breakdown above.")

## 4 — Step 1: raw mosaic, both objectives together

Composites every tile (both objectives, UNSHIFTED) into one image -- since
the high-mag calibration tiles' own `zvalue` is always higher than every
low-mag tile's (Steve's own acquisition-order stacking, confirmed directly
from the loaded tiles), the high-mag patch paints on top wherever they
overlap. Whether the two objectives actually agree on where the tissue is
should be visible right away, from real tissue content -- not just tile
CENTER coordinates, which are indistinguishable at this whole-mosaic scale
for a shift of only a few hundred um.

The two objectives were shot at very different exposures -- composited at
their own raw values, the high-mag patch is badly saturated relative to
the low-mag background, which would make the comparison unreadable.
`normalize_tile_intensities` below independently rescales each objective's
own tiles onto a shared display range first (`LOW_MAG_DISPLAY_PCT`/
`HIGH_MAG_DISPLAY_PCT`, section 2) -- a DISPLAY-only transform; segmentation
in step 4 below reads the real, un-rescaled low-mag pixel values.

In [ ]:
low_tiles  = [t for t in tiles_all if t.objective_name == LOW_MAG_OBJECTIVE]
high_tiles = [t for t in tiles_all if t.objective_name == HIGH_MAG_OBJECTIVE]


def normalize_tile_intensities(tiles, low_pct, high_pct, target_lo=0.0, target_hi=1000.0):
    # Rescale every tile's own raw pixel values from this GROUP's pooled
    # [low_pct, high_pct] percentile range onto [target_lo, target_hi] --
    # display-only (returns new tile copies; originals untouched), so tiles
    # shot at very different exposures composite onto one comparable scale.
    sample = np.concatenate([t.image[::4, ::4].ravel() for t in tiles])
    lo, hi = np.percentile(sample, [low_pct, high_pct])
    if hi <= lo:
        return list(tiles)
    rescaled = []
    for t in tiles:
        norm = np.clip((t.image.astype(np.float32) - lo) / (hi - lo), 0.0, 1.0)
        rescaled.append(dataclasses.replace(t, image=norm * (target_hi - target_lo) + target_lo))
    return rescaled


low_tiles_norm  = normalize_tile_intensities(low_tiles,  *LOW_MAG_DISPLAY_PCT)
high_tiles_norm = normalize_tile_intensities(high_tiles, *HIGH_MAG_DISPLAY_PCT)

raw_both_canvas = assemble_mosaic_canvas(low_tiles_norm + high_tiles_norm, working_pixel_um=WORKING_PIXEL_UM)

fig, ax = plt.subplots(figsize=(10, 10))
ax.imshow(raw_both_canvas.image, cmap="gray", vmin=0, vmax=1000)
ax.set_title(f"{SAMPLE_NAME}: raw mosaic, both objectives (uncorrected)")
ax.axis("off")
fig.tight_layout()
fig.savefig(FIGURES_DIR / f"{NOTEBOOK_NAME}.step1_raw_mosaic_both_objectives.png", dpi=150)
plt.show()
print(f"Saved: {FIGURES_DIR / f'{NOTEBOOK_NAME}.step1_raw_mosaic_both_objectives.png'}")

## 5 — Step 2: shifted mosaic, both objectives together

Same composite as step 1, but with the known `(SHIFT_DX_UM, SHIFT_DY_UM)`
correction applied to every LOW-MAG tile's position first (the high-mag
tiles are the fixed calibration reference and never move). If the shift is
right, the high-mag patch should now look like a seamless, continuous part
of the surrounding low-mag tissue instead of a visibly separate/offset
patch.

In [ ]:
low_tiles_shifted = [dataclasses.replace(t, x_um=t.x_um + SHIFT_DX_UM, y_um=t.y_um + SHIFT_DY_UM)
                     for t in low_tiles]
low_tiles_shifted_norm = normalize_tile_intensities(low_tiles_shifted, *LOW_MAG_DISPLAY_PCT)

shifted_both_canvas = assemble_mosaic_canvas(low_tiles_shifted_norm + high_tiles_norm, working_pixel_um=WORKING_PIXEL_UM)

fig, ax = plt.subplots(figsize=(10, 10))
ax.imshow(shifted_both_canvas.image, cmap="gray", vmin=0, vmax=1000)
ax.set_title(f"{SAMPLE_NAME}: shifted mosaic, both objectives (dX={SHIFT_DX_UM}, dY={SHIFT_DY_UM} um)")
ax.axis("off")
fig.tight_layout()
fig.savefig(FIGURES_DIR / f"{NOTEBOOK_NAME}.step2_shifted_mosaic_both_objectives.png", dpi=150)
plt.show()
print(f"Saved: {FIGURES_DIR / f'{NOTEBOOK_NAME}.step2_shifted_mosaic_both_objectives.png'}")
print("Visually confirm the high-mag patch now looks like a seamless part of the "
      "surrounding low-mag tissue (compare against step1's unshifted version) before "
      "trusting anything below -- if not, SHIFT_DX_UM/SHIFT_DY_UM (or their sign) "
      "need correcting. If in doubt, flag it rather than guessing.")

### Zoomed: just the tissue region covered by the high-mag FOVs, before vs. after

The whole-mosaic composites above are too coarse to show a few-hundred-um
shift directly (tile spacing is much larger than the shift). Cropped to the
high-mag tiles' own bounding box instead (+ a small margin of low-mag
context), at a finer `ZOOM_PIXEL_UM` than the whole-mosaic canvas, side by
side before vs. after -- both objectives still independently intensity-
normalized (`normalize_tile_intensities`, same as above) onto one shared
display scale.

In [ ]:
ZOOM_PIXEL_UM  = 1.5    # finer than WORKING_PIXEL_UM -- this crop is small, so it's affordable
ZOOM_MARGIN_UM = 400.0  # extra low-mag context around the high-mag tiles' own bounding box

high_xs = np.array([t.x_um for t in high_tiles]); high_ys = np.array([t.y_um for t in high_tiles])
patch_half_um = high_tiles[0].image.shape[0] * high_tiles[0].pixel_size_um / 2
zoom_x0, zoom_x1 = high_xs.min() - patch_half_um - ZOOM_MARGIN_UM, high_xs.max() + patch_half_um + ZOOM_MARGIN_UM
zoom_y0, zoom_y1 = high_ys.min() - patch_half_um - ZOOM_MARGIN_UM, high_ys.max() + patch_half_um + ZOOM_MARGIN_UM

def _tile_overlaps_zoom(t):
    half = (t.image.shape[1] * t.pixel_size_um / 2, t.image.shape[0] * t.pixel_size_um / 2)
    return not (t.x_um + half[0] < zoom_x0 or t.x_um - half[0] > zoom_x1
                or t.y_um + half[1] < zoom_y0 or t.y_um - half[1] > zoom_y1)

low_near_zoom_before = [t for t in low_tiles         if _tile_overlaps_zoom(t)]
low_near_zoom_after  = [t for t in low_tiles_shifted if _tile_overlaps_zoom(t)]
print(f"Low-mag tiles near the high-mag patch: {len(low_near_zoom_before)} (before), "
      f"{len(low_near_zoom_after)} (after)")

low_near_zoom_before_norm = normalize_tile_intensities(low_near_zoom_before, *LOW_MAG_DISPLAY_PCT)
low_near_zoom_after_norm  = normalize_tile_intensities(low_near_zoom_after,  *LOW_MAG_DISPLAY_PCT)

zoom_canvas_before = assemble_mosaic_canvas(low_near_zoom_before_norm + high_tiles_norm, working_pixel_um=ZOOM_PIXEL_UM)
zoom_canvas_after  = assemble_mosaic_canvas(low_near_zoom_after_norm  + high_tiles_norm, working_pixel_um=ZOOM_PIXEL_UM)

fig, axes = plt.subplots(1, 2, figsize=(15, 7.5))
for ax, canvas, title in (
    (axes[0], zoom_canvas_before, "before shift"),
    (axes[1], zoom_canvas_after,  f"after shift (dX={SHIFT_DX_UM}, dY={SHIFT_DY_UM} um)"),
):
    ax.imshow(canvas.image, cmap="gray", vmin=0, vmax=1000)
    ax.set_title(title); ax.axis("off")
fig.suptitle(f"{SAMPLE_NAME}: zoomed to the high-mag FOVs' own bounding box")
fig.tight_layout()
fig.savefig(FIGURES_DIR / f"{NOTEBOOK_NAME}.step2_zoomed_before_after.png", dpi=150)
plt.show()
print(f"Saved: {FIGURES_DIR / f'{NOTEBOOK_NAME}.step2_zoomed_before_after.png'}")

## 6 — Step 3: shifted mosaic canvas + current (already-imaging) positions overlay

Assembles the mosaic from the SHIFTED low-mag tiles only, at their own REAL
(un-normalized) pixel values (same `MOSAIC_KEEP_OBJECTIVES`/
`WORKING_PIXEL_UM` convention as the local `02_create_boundary_from_
mosaic.ipynb` -- this canvas also feeds segmentation in step 4), then
overlays every FOV in the positions file currently being imaged as its own
real PERIMETER square (not just a center point) -- if the original
boundary really was derived from the unshifted mosaic, this should show
the current FOV grid sitting offset from the real tissue.

In [ ]:
shifted_canvas = assemble_mosaic_canvas(
    [t for t in low_tiles_shifted if t.objective_name in MOSAIC_KEEP_OBJECTIVES],
    working_pixel_um=WORKING_PIXEL_UM,
)

current_positions = np.loadtxt(CURRENT_POSITIONS_PATH, delimiter=",")
print(f"Current positions: {len(current_positions)} FOV(s) from {CURRENT_POSITIONS_PATH.name}")

def _um_to_px(canvas, x, y):
    return ((np.asarray(x) - canvas.origin_um[0]) / canvas.pixel_size_um,
             (np.asarray(y) - canvas.origin_um[1]) / canvas.pixel_size_um)

fig, ax = plt.subplots(figsize=(9, 9))
covered_vals = shifted_canvas.image[shifted_canvas.covered]
vmin, vmax = np.percentile(covered_vals, [1, 99]) if covered_vals.size else (0, 1)
ax.imshow(shifted_canvas.image, cmap="gray", vmin=vmin, vmax=vmax)

half_px = (fov_size_um / 2) / shifted_canvas.pixel_size_um
px, py = _um_to_px(shifted_canvas, current_positions[:, 0], current_positions[:, 1])
for x, y in zip(px, py):
    ax.add_patch(mpatches.Rectangle((x - half_px, y - half_px), 2 * half_px, 2 * half_px,
                                     linewidth=0.4, edgecolor="yellow", facecolor="none"))
ax.plot([], [], color="yellow", label=f"current positions ({len(current_positions)})")
ax.set_title(f"{SAMPLE_NAME}: shifted (corrected) mosaic vs. currently-imaging FOV grid")
ax.legend(); ax.axis("off")
fig.tight_layout()
fig.savefig(FIGURES_DIR / f"{NOTEBOOK_NAME}.step3_shifted_mosaic_vs_current_fovs.png", dpi=150)
plt.show()
print(f"Saved: {FIGURES_DIR / f'{NOTEBOOK_NAME}.step3_shifted_mosaic_vs_current_fovs.png'}")

## 7 — Step 4: re-segment the shifted mosaic, find FOVs using the SAME grid as OLD

Same `segment_mosaic_tissue` parameters as the local `02_create_boundary_
from_mosaic.ipynb`. For the FOV grid, this does NOT build an independent
grid centered on the new (shifted) tissue polygon's own bounding box --
`create_grid_positions` centers a fresh grid on whatever polygon it's given,
and the shift (~2.25 x 2.31 FOV widths) isn't an exact integer number of
FOVs, so an independently-centered new grid would share almost no exact
coordinates with the current positions file even where they really overlap
(forcing an approximate, distance-based comparison). Instead: build ONE
dense grid centered on the OLD tissue boundary's own bounding box (the
EXACT phase the current positions file's own grid already uses), sized to
cover both the old and shifted/new tissue regions, then FILTER that same
dense grid twice -- once against the OLD boundary+holes (sanity check:
should reproduce `current_positions` count almost exactly) and once against
the NEW (shifted) boundary+holes. Any FOV the two filtered sets share is
then an EXACT coordinate match, not an approximate overlap -- holes are
loaded from the EXISTING `positions/boundaries/from_mosaic/hole*.txt` files
(holes don't move; only the outer tissue boundary was ever derived from the
un-shifted mosaic's own threshold trace) and shifted the same way.

In [ ]:
BOUNDARY_DIR = POSITIONS_DIR / "boundaries" / "from_mosaic"

old_boundary_polygon = load_boundary_polygon(BOUNDARY_DIR / "boundary_positions.txt")
holes_raw = load_hole_polygons(BOUNDARY_DIR)
holes_shifted = [translate(h, xoff=SHIFT_DX_UM, yoff=SHIFT_DY_UM) for h in holes_raw]
print(f"Loaded OLD boundary + {len(holes_raw)} hole polygon(s)")

segmentation = segment_mosaic_tissue(
    shifted_canvas,
    threshold           = THRESHOLD,
    smooth_sigma_um     = SMOOTH_SIGMA_UM,
    close_radius_um     = CLOSE_RADIUS_UM,
    open_radius_um      = OPEN_RADIUS_UM,
    margin_um           = MARGIN_UM,
    min_tissue_area_um2 = MIN_TISSUE_AREA_UM2,
    min_hole_area_um2   = MIN_HOLE_AREA_UM2,
    min_island_area_um2 = MIN_ISLAND_AREA_UM2,
    simplify_tol_um     = SIMPLIFY_TOL_UM,
)
print(f"Threshold used : {segmentation.threshold:.1f}")
print(f"Tissue pieces  : {len(segmentation.tissue_polygons)}")

if len(segmentation.tissue_polygons) != 1:
    raise ValueError(
        f"Expected exactly 1 tissue piece for this sample's known single-boundary "
        f"layout, got {len(segmentation.tissue_polygons)} -- inspect the plot below "
        f"and adjust THRESHOLD/morphology parameters before continuing."
    )
new_tissue_polygon = segmentation.tissue_polygons[0]

ax = plot_mosaic_segmentation(shifted_canvas, segmentation)
ax.figure.savefig(FIGURES_DIR / f"{NOTEBOOK_NAME}.step4_new_segmentation.png", dpi=150)
plt.show()

# ONE dense grid, phase-locked to the OLD boundary's own bounding-box center,
# sized to cover both the old and new (shifted) tissue regions.
xmin_old, ymin_old, xmax_old, ymax_old = old_boundary_polygon.bounds
cx_old, cy_old = (xmin_old + xmax_old) / 2.0, (ymin_old + ymax_old) / 2.0
xmin_new, ymin_new, xmax_new, ymax_new = new_tissue_polygon.bounds
half_w = max(xmax_old - cx_old, xmax_new - cx_old, cx_old - xmin_old, cx_old - xmin_new)
half_h = max(ymax_old - cy_old, ymax_new - cy_old, cy_old - ymin_old, cy_old - ymin_new)
grid_extent_polygon = box(cx_old - half_w, cy_old - half_h, cx_old + half_w, cy_old + half_h)

grid, _, _ = create_grid_positions(grid_extent_polygon, step_size_um, direction=SCAN_DIRECTION)
dense_path = generate_scanning_path(grid, direction=SCAN_DIRECTION)

old_grid_positions = filter_scanning_path(dense_path, old_boundary_polygon, holes_raw, fov_size_um)
new_grid_positions = filter_scanning_path(dense_path, new_tissue_polygon, holes_shifted, fov_size_um)

print(f"Old boundary, from the shared grid : {len(old_grid_positions)} FOV(s) "
      f"(current positions file has {len(current_positions)})")
print(f"New (shifted) boundary, from the shared grid: {len(new_grid_positions)} FOV(s)")

## 8 — Step 5: overlay the OLD vs. NEW tissue boundary outlines

In [ ]:
fig, ax = plt.subplots(figsize=(9, 9))
ax.plot(*old_boundary_polygon.exterior.xy, "-",  lw=1.2, c="tab:orange", label="OLD boundary")
ax.plot(*new_tissue_polygon.exterior.xy,   "--", lw=1.2, c="tab:green",  label="NEW (shifted) boundary")
for h in holes_raw:
    ax.plot(*h.exterior.xy, "-", lw=0.6, c="0.6")
for h in holes_shifted:
    ax.plot(*h.exterior.xy, "--", lw=0.6, c="0.4")
ax.set_title(f"{SAMPLE_NAME}: OLD vs. NEW tissue boundary")
ax.legend(); ax.axis("equal"); ax.invert_yaxis()
ax.set_xlabel("stage x (um)"); ax.set_ylabel("stage y (um)")
fig.tight_layout()
fig.savefig(FIGURES_DIR / f"{NOTEBOOK_NAME}.step5_old_vs_new_boundary.png", dpi=150)
plt.show()
print(f"Saved: {FIGURES_DIR / f'{NOTEBOOK_NAME}.step5_old_vs_new_boundary.png'}")

## 9 — Step 6: classify FOVs as already-covered, MISSING, or UNNECESSARY

Since `old_grid_positions` and `new_grid_positions` (section 7) come from
filtering the exact SAME dense grid, a FOV present in both is an EXACT
coordinate match -- no distance-based fuzzy matching needed.
- **MISSING** = in the NEW (shifted) boundary's filtered set but NOT the
  OLD's: real tissue area the old (mis-positioned) grid never imaged.
- **UNNECESSARY** = in the OLD boundary's filtered set but NOT the NEW's:
  FOVs the old grid DID image, but that fall outside the corrected tissue
  boundary -- imaging time spent on non-tissue/off-target area.
All drawn as real FOV perimeter squares over both boundary outlines, not
center points.

In [ ]:
def _coord_set(coords, decimals=3):
    return {(round(float(x), decimals), round(float(y), decimals)) for x, y in coords}


old_set = _coord_set(old_grid_positions)
new_set = _coord_set(new_grid_positions)

missing_mask     = np.array([tuple(np.round(p, 3)) not in old_set for p in new_grid_positions])
unnecessary_mask = np.array([tuple(np.round(p, 3)) not in new_set for p in old_grid_positions])

missing_coords_unordered = new_grid_positions[missing_mask]
unnecessary_coords       = old_grid_positions[unnecessary_mask]

print(f"NEW (shifted boundary) FOVs             : {len(new_grid_positions)}")
print(f"Already covered (exact match with OLD)  : {(~missing_mask).sum()}")
print(f"MISSING (no exact match in OLD)         : {missing_mask.sum()}")
print(f"OLD (current) FOVs                      : {len(old_grid_positions)}")
print(f"UNNECESSARY (no exact match in NEW)      : {unnecessary_mask.sum()}")

fig, ax = plt.subplots(figsize=(9, 9))
ax.plot(*old_boundary_polygon.exterior.xy, "-",  lw=1, c="0.5",      label="OLD boundary")
ax.plot(*new_tissue_polygon.exterior.xy,   "--", lw=1, c="tab:blue", label="NEW boundary")
for h in holes_raw:
    ax.plot(*h.exterior.xy, "-", lw=0.5, c="0.7")

half_um = fov_size_um / 2
for x, y in new_grid_positions[~missing_mask]:
    ax.add_patch(mpatches.Rectangle((x - half_um, y - half_um), fov_size_um, fov_size_um,
                                     linewidth=0.3, edgecolor="tab:green", facecolor="none"))
for x, y in missing_coords_unordered:
    ax.add_patch(mpatches.Rectangle((x - half_um, y - half_um), fov_size_um, fov_size_um,
                                     linewidth=0.4, edgecolor="tab:red", facecolor="tab:red", alpha=0.3))
for x, y in unnecessary_coords:
    ax.add_patch(mpatches.Rectangle((x - half_um, y - half_um), fov_size_um, fov_size_um,
                                     linewidth=0.4, edgecolor="tab:purple", facecolor="tab:purple", alpha=0.3))
ax.plot([], [], color="tab:green",  label=f"already covered ({(~missing_mask).sum()})")
ax.plot([], [], color="tab:red",    label=f"MISSING ({missing_mask.sum()})")
ax.plot([], [], color="tab:purple", label=f"UNNECESSARY ({unnecessary_mask.sum()})")
ax.set_title(f"{SAMPLE_NAME}: missing vs. unnecessary FOVs")
ax.legend(); ax.axis("equal"); ax.invert_yaxis()
ax.set_xlabel("stage x (um)"); ax.set_ylabel("stage y (um)")
fig.tight_layout()
fig.savefig(FIGURES_DIR / f"{NOTEBOOK_NAME}.step6_missing_fovs.png", dpi=150)
plt.show()
print(f"Saved: {FIGURES_DIR / f'{NOTEBOOK_NAME}.step6_missing_fovs.png'}")

## 10 — Old-to-new FOV renumbering table

An alternative to appending the missing FOVs at the end of the CURRENT
positions file (sections 11-12 below): image every REMAINING round
directly from the NEW (corrected) positions file instead. That changes
what FOV INDEX corresponds to a given physical location -- rounds already
imaged used the OLD numbering (row index in the current positions file);
any new round would use the NEW numbering (row index in `new_grid_
positions`, i.e. its own boustrophedon order over the corrected boundary).
To combine already-imaged rounds with newly-imaged ones later (e.g. during
MERlin decoding, which assumes one consistent `fov` index across all
rounds), every old FOV needs to be mapped to its new index -- or flagged as
dropped if it fell outside the corrected boundary (the UNNECESSARY FOVs
from section 9, never useful for decoding anyway since they're off-tissue).

`current_positions`' own row order is used directly as the "old" side of
the mapping below (not `old_grid_positions`'s order), since it's the real
numbering already baked into every already-acquired round's own file
names. Verified below that the two are the exact same SET of coordinates
(confirmed on real LT060_sample_04 data: identical, zero in one but not
the other) -- their ROW ORDER does typically differ, though, since
`old_grid_positions` here is built from a plain `generate_scanning_path`
without also applying `close_scanning_path` (the return-leg reordering
step this sample's real positions file was originally built with via
`build_boundary_path(..., return_side="top")`) -- irrelevant for every
SET-based comparison this notebook does, but worth not mistaking for a
real discrepancy.

In [ ]:
assert len(current_positions) == len(old_grid_positions), (
    f"current_positions has {len(current_positions)} FOV(s) but re-deriving from the "
    f"OLD boundary against the shared grid gives {len(old_grid_positions)} -- the "
    f"renumbering below assumes these match exactly."
)
current_set = _coord_set(current_positions)
old_grid_set = _coord_set(old_grid_positions)
print(f"current_positions vs. old_grid_positions: {len(current_positions)} FOV(s) each, "
      f"same SET of coordinates: {current_set == old_grid_set} "
      f"(in current only: {len(current_set - old_grid_set)}, in old_grid only: {len(old_grid_set - current_set)})")

def _round_tuple(p, decimals=3):
    return (round(float(p[0]), decimals), round(float(p[1]), decimals))

new_index_by_coord = {_round_tuple(p): i for i, p in enumerate(new_grid_positions)}

renumber_rows = []
for old_idx, (x, y) in enumerate(current_positions):
    new_idx = new_index_by_coord.get(_round_tuple((x, y)))
    renumber_rows.append({
        "old_fov_index": old_idx,
        "old_x_um": x, "old_y_um": y,
        "new_fov_index": new_idx if new_idx is not None else -1,
        "kept": new_idx is not None,
    })
renumber_df = pd.DataFrame(renumber_rows)

RENUMBER_PATH = POSITIONS_DIR / f"fov_renumbering_{POSITIONS_TAG}.csv"
renumber_df.to_csv(RENUMBER_PATH, index=False)

print(f"Kept (old FOV also in the new/corrected grid): {renumber_df['kept'].sum()}")
print(f"Dropped (old FOV is UNNECESSARY, not in new)  : {(~renumber_df['kept']).sum()}")
print(f"Saved: {RENUMBER_PATH}")
renumber_df.head(20)

## 11 — Step 7: re-order the MISSING FOVs into their own short-travel loop

Simply keeping `new_grid_positions`'s own boustrophedon order (inherited
from `dense_path`, section 7) for JUST the missing subset does NOT give a
sensible loop: that order snakes through the WHOLE tissue column by
column, and any single column can contribute anywhere from zero to all of
its FOVs to the missing set depending on where that column's own boundary
happens to differ from the old grid -- extracting a boolean-masked
subsequence from that full snake jumps unpredictably between whichever
fragments happen to be missing in each column, in column order, not in a
locally-short path.

Tried a simple per-column boustrophedon re-sort first (group by real
lattice column, sort each column by the other axis, alternate direction
column to column -- the same convention `generate_scanning_path` uses for
a full grid). Measuring actual total travel (`get_path_stats`) showed this
was NOT reliably better than the naive subsequence order: a single lattice
column can itself contain more than one disconnected run of missing FOVs
(e.g. two separate notches crossing the same column), and sorting that
column's points by one axis alone still jumps across the gap between runs.
**Used a greedy nearest-neighbor walk instead** -- starting from the
missing FOV closest to `current_positions`'s own last point (so the
transit from the existing loop's end into this new loop is also short, not
just the loop's own internal travel), then repeatedly stepping to the
nearest not-yet-visited missing FOV. Simple and never takes a large hop
when a smaller one is available; not a guaranteed-optimal tour (true
optimal touring is NP-hard), but directly measured below to confirm it
beats both alternatives on this real data before using it.

In [ ]:
def nearest_neighbor_order(coords, start_point):
    remaining = coords.copy()
    order = []
    current = np.asarray(start_point, dtype=float)
    while len(remaining):
        dists = np.linalg.norm(remaining - current, axis=1)
        idx = int(np.argmin(dists))
        current = remaining[idx]
        order.append(current)
        remaining = np.delete(remaining, idx, axis=0)
    return np.array(order)


def boustrophedon_order(coords, step_size_um, direction="vertical"):
    primary_axis, secondary_axis = (0, 1) if direction == "vertical" else (1, 0)
    col_idx = np.round((coords[:, primary_axis] - coords[:, primary_axis].min()) / step_size_um).astype(int)
    chunks = []
    for col in sorted(set(col_idx)):
        members = coords[col_idx == col]
        members = members[np.argsort(members[:, secondary_axis])]
        if col % 2 == 1:
            members = members[::-1]
        chunks.append(members)
    return np.concatenate(chunks, axis=0)


missing_coords_column_sorted = boustrophedon_order(missing_coords_unordered, step_size_um, direction=SCAN_DIRECTION)
missing_coords_nearest_neighbor = nearest_neighbor_order(missing_coords_unordered, current_positions[-1])

candidates = {
    "new_grid_positions' own filtered order": missing_coords_unordered,
    "per-column boustrophedon re-sort":       missing_coords_column_sorted,
    "greedy nearest-neighbor walk":           missing_coords_nearest_neighbor,
}
for label, coords in candidates.items():
    length, max_step = get_path_stats(coords)
    print(f"{label:38s}: {length/1000:6.2f} mm total (max single step {max_step:6.0f} um)")

missing_coords = missing_coords_nearest_neighbor
print(f"\nUsing: greedy nearest-neighbor walk")

## 12 — Step 8-9: append the re-ordered MISSING FOVs, save a new positions file

Appended directly after the current (OLD) positions, so the existing
imaging loop's own order is completely undisturbed; only a new loop over
the appended tail needs to be added in Dave.

Writes a NEW file (`_added` suffix) -- never overwrites the original
positions file this sample is currently being imaged from.

In [ ]:
added_positions = np.concatenate([current_positions, missing_coords], axis=0)
save_positions_array(added_positions, ADDED_POSITIONS_PATH)

print(f"Wrote {len(added_positions)} FOV(s) ({len(current_positions)} original + "
      f"{len(missing_coords)} appended missing) to:")
print(f"  {ADDED_POSITIONS_PATH}")

fig, ax = plt.subplots(figsize=(9, 9))
ax.scatter(current_positions[:, 0], current_positions[:, 1],
           s=6, c="tab:orange", label=f"original ({len(current_positions)})")
ax.scatter(missing_coords[:, 0], missing_coords[:, 1],
           s=10, c="tab:red", label=f"appended, missing ({len(missing_coords)})")
ax.plot(missing_coords[:, 0], missing_coords[:, 1], "-", lw=0.5, c="tab:red", alpha=0.5)
ax.set_title(f"{SAMPLE_NAME}: final positions file ({ADDED_POSITIONS_PATH.name})")
ax.legend(); ax.axis("equal"); ax.invert_yaxis()
ax.set_xlabel("stage x (um)"); ax.set_ylabel("stage y (um)")
fig.tight_layout()
fig.savefig(FIGURES_DIR / f"{NOTEBOOK_NAME}.step7_final_added_positions.png", dpi=150)
plt.show()
print(f"Saved: {FIGURES_DIR / f'{NOTEBOOK_NAME}.step7_final_added_positions.png'}")
print()
print("Review every figure in", FIGURES_DIR, "before using this file -- in particular "
      "step2 (does the shift actually align the two objectives?) and step6/7 (do the "
      "MISSING FOVs look like a real tissue-edge strip, not noise).")

## 13 — How many OLD FOVs are empty vs. have real tissue signal?

Reads each of the `current_positions` FOVs' own cells-round frame once
(cached locally per FOV -- `analysis/cache/fix_mosaic_shift_missing_fovs/
old_fov_cells_stats.csv`, so a re-run doesn't re-read 1166 raw files),
computes a robust per-FOV signal statistic (`SIGNAL_PERCENTILE`, the
frame's own 99th-percentile intensity -- more sensitive to a small bright
nuclei population than the mean, which a mostly-empty frame with a few
bright cells would wash out), and classifies empty vs. has-cells from a
threshold estimated from the pooled histogram's own bimodal structure
(`acquisition.mosaic._estimate_bimodal_threshold` -- the same valley-
between-two-peaks estimator `02_create_boundary_from_mosaic.ipynb` uses
for its own tissue/background split, reused here on a per-FOV signal
distribution instead of per-pixel mosaic intensities). Shown as a histogram
below for visual confirmation before trusting it -- override
`EMPTY_THRESHOLD` manually and re-run the classification cell if it looks
wrong.

In [ ]:
from MERci.acquisition.mosaic import _estimate_bimodal_threshold

round_info_df = pd.read_csv(SAMPLE_DIR / "metadata" / "round_info.csv")
cells_row = round_info_df[round_info_df["imaging_type"].str.strip().str.lower() == "cells"].iloc[0]
CELLS_ROUND_ID   = int(cells_row["imaging_round"])
CELLS_SERIES_PATTERN = cells_row["series"]   # e.g. "hal-st2-cells_{fov:04d}"

# data_dir's own dir column can be a stale absolute path from a different
# machine (this sample's is a now-unreachable "U:\..." one) -- resolve
# relative to THIS machine's own (confirmed-reachable) SAMPLE_DIR instead,
# matching where "cells" data actually lives for every layout this repo
# supports (top-level data/ or a data/cells/ subfolder).
cells_data_dir = SAMPLE_DIR / "data" / "cells"
if not cells_data_dir.exists():
    cells_data_dir = SAMPLE_DIR / "data"

cells_hal_config_path = SAMPLE_DIR / "settings" / cells_row["hal_config"]
cells_frame_table_path = find_frame_table_for_hal_config(cells_hal_config_path, SAMPLE_DIR / "metadata")
if cells_frame_table_path is None:
    raise FileNotFoundError(f"Could not find the frame table for {cells_hal_config_path}.")
cells_frame_table = pd.read_csv(cells_frame_table_path, index_col=0)
cells_color_frames = get_color_frame_indices(cells_frame_table)
CELLS_COLOR_NM = 405.0 if 405.0 in cells_color_frames else next(iter(cells_color_frames))
CELLS_FRAME_IDX = cells_color_frames[CELLS_COLOR_NM]
print(f"Cells round: {CELLS_ROUND_ID}, color {CELLS_COLOR_NM:.0f} nm, frame {CELLS_FRAME_IDX}, "
      f"data dir: {cells_data_dir}")

SIGNAL_PERCENTILE = 99.0
old_stats_cache = CACHE_DIR / "old_fov_cells_stats.csv"

if old_stats_cache.exists():
    old_stats_df = pd.read_csv(old_stats_cache)
    print(f"Loaded cached per-FOV stats: {old_stats_cache}")
else:
    rows = []
    reporter = ProgressReporter(total=len(current_positions), label="Reading OLD FOVs' cells frame")
    for old_idx in reporter.wrap(range(len(current_positions))):
        fpath = cells_data_dir / (CELLS_SERIES_PATTERN.format(fov=old_idx) + IMAGE_SUFFIX)
        if not fpath.exists():
            rows.append({"fov_id": old_idx, "signal_pctl": np.nan})
            continue
        frame = read_image_frames(fpath, [CELLS_FRAME_IDX])[0]
        rows.append({"fov_id": old_idx, "signal_pctl": float(np.percentile(frame, SIGNAL_PERCENTILE))})
    old_stats_df = pd.DataFrame(rows)
    old_stats_df.to_csv(old_stats_cache, index=False)
    print(f"Cached: {old_stats_cache}")

valid = old_stats_df["signal_pctl"].notna()
print(f"{valid.sum()}/{len(old_stats_df)} FOV(s) had a readable cells-round file.")

signal_vals = old_stats_df.loc[valid, "signal_pctl"].values
log_vals = np.log10(np.clip(signal_vals, 1, None))
hist_counts, bin_edges = np.histogram(log_vals, bins=60)
bin_centers_log = (bin_edges[:-1] + bin_edges[1:]) / 2
EMPTY_THRESHOLD = _estimate_bimodal_threshold(bin_centers_log, hist_counts)
if EMPTY_THRESHOLD is None:
    # The median is a bad fallback here: a real, sharp "empty" spike sitting
    # next to a broad, unevenly-shaped "has tissue" distribution (confirmed
    # directly on real LT060_sample_04 data) makes the median land WELL
    # inside the tissue distribution itself, producing a meaningless ~50/50
    # split rather than the real narrow-spike/broad-hump valley -- the two
    # classes' very different SIZES (far fewer empty FOVs than tissue-
    # containing ones here) throw off simple valley-between-two-peaks
    # detection. Otsu's method (maximizes between-class variance directly,
    # not peak-shape-dependent) handles this specific shape far better.
    from skimage.filters import threshold_otsu
    EMPTY_THRESHOLD = float(10 ** threshold_otsu(log_vals))
    print(f"WARNING: no clear bimodal split found by peak/valley detection -- "
          f"falling back to Otsu's method ({EMPTY_THRESHOLD:.1f}); inspect the "
          f"histogram below and override EMPTY_THRESHOLD manually if it looks wrong.")

fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(signal_vals, bins=60, color="tab:blue")
ax.axvline(EMPTY_THRESHOLD, color="r", ls="--", label=f"EMPTY_THRESHOLD={EMPTY_THRESHOLD:.1f}")
ax.set_xlabel(f"FOV {SIGNAL_PERCENTILE:.0f}th-percentile intensity (cells round, {CELLS_COLOR_NM:.0f} nm)")
ax.set_ylabel("FOV count")
ax.legend()
ax.set_title(f"{SAMPLE_NAME}: OLD FOV signal histogram")
fig.tight_layout()
fig.savefig(FIGURES_DIR / f"{NOTEBOOK_NAME}.step8_old_fov_signal_histogram.png", dpi=150)
plt.show()
print(f"Saved: {FIGURES_DIR / f'{NOTEBOOK_NAME}.step8_old_fov_signal_histogram.png'}")
print("Review this histogram before trusting the counts below -- confirm EMPTY_THRESHOLD "
      "actually sits in the valley between two real peaks, not an artifact.")

In [ ]:
old_stats_df["has_cells"] = old_stats_df["signal_pctl"] >= EMPTY_THRESHOLD
n_empty     = int((~old_stats_df["has_cells"] & valid).sum())
n_has_cells = int((old_stats_df["has_cells"] & valid).sum())

print(f"OLD FOVs (valid reads)  : {int(valid.sum())}")
print(f"  empty                : {n_empty} ({n_empty / valid.sum():.1%})")
print(f"  has real tissue signal: {n_has_cells} ({n_has_cells / valid.sum():.1%})")

## 14 — NEW/MISSING FOVs: percentage empty vs. percentage with real signal

MISSING FOVs (section 9) were never actually imaged, so there's no real
per-FOV cells-round data for them. As a proxy, sample the mean LOW-MAG
MOSAIC intensity (`shifted_canvas`, the exact canvas segmentation itself
reads in section 7) within each missing FOV's own real footprint, and
classify against the SAME `THRESHOLD` (section 2) already used for tissue
segmentation -- consistent with how the tissue boundary itself was drawn,
rather than a newly-invented cutoff.

In [ ]:
def _fov_mean_mosaic_intensity(canvas, x_um, y_um, fov_size_um):
    half_px = (fov_size_um / 2) / canvas.pixel_size_um
    col = (x_um - canvas.origin_um[0]) / canvas.pixel_size_um
    row = (y_um - canvas.origin_um[1]) / canvas.pixel_size_um
    r0, r1 = max(0, int(row - half_px)), min(canvas.image.shape[0], int(row + half_px))
    c0, c1 = max(0, int(col - half_px)), min(canvas.image.shape[1], int(col + half_px))
    patch = canvas.image[r0:r1, c0:c1]
    return float(patch.mean()) if patch.size else 0.0


missing_mosaic_intensity = np.array([
    _fov_mean_mosaic_intensity(shifted_canvas, x, y, fov_size_um)
    for x, y in missing_coords
])
missing_is_full = missing_mosaic_intensity >= THRESHOLD
n_missing_full  = int(missing_is_full.sum())
n_missing_empty = int((~missing_is_full).sum())

print(f"MISSING FOVs: {len(missing_coords)}")
print(f"  empty (mosaic intensity below THRESHOLD={THRESHOLD}): "
      f"{n_missing_empty} ({n_missing_empty / len(missing_coords):.1%})")
print(f"  full  (mosaic intensity at/above THRESHOLD={THRESHOLD}): "
      f"{n_missing_full} ({n_missing_full / len(missing_coords):.1%})")

## 15 — If hybs 1-5 (bits 1-10) are lost, which LT2 genes are affected?

Parses `data/configs/merlin/codebooks/LT2v0_codebook.csv` using MERlin's
own codebook-loading logic (`merlin.data.codebook.Codebook.__init__`'s
OLD-FORMAT branch -- this file's header starts with `version`, so a plain
`pandas.read_csv` fails to parse it; the header-skip + per-row barcode-
string parsing below is copied from that real source, not guessed).
Verified directly from the real codebook (not assumed): every one of its
308 barcodes (258 genes + 50 blanks) has exactly 4 "on" bits, and the
minimum pairwise Hamming distance across all of them is 4 -- a genuine
MHD4 (Modified Hamming Distance 4) code, matching `merlin.util.decoding.
PixelBasedDecoder`'s nearest-neighbor-by-normalized-Euclidean-distance
scheme (`distanceThreshold=0.5176` default -- calibrated for exactly this
kind of code). Since any two codewords differ in at least 4 bit positions,
a barcode with exactly ONE of its 4 "on" bits dropped to 0 (imaged in a
now-dead channel) is closer to its TRUE codeword (Hamming distance 1) than
to any OTHER codeword (Hamming distance >= 4 - 1 = 3 by the triangle
inequality) -- MERFISH's standard single-error correction still recovers
it. A barcode with TWO OR MORE of its 4 "on" bits inside the dead-bit range
has no such guarantee and is treated as lost.

In [ ]:
import csv as _csv

CODEBOOK_PATH = MERCI_DIR / "data" / "configs" / "merlin" / "codebooks" / "LT2v0_codebook.csv"
DEAD_BITS = list(range(1, 11))   # hybs 1-5 -> bits 1-10, per metadata/round_bit_color_map.csv

def _parse_barcode_from_string(s):
    return np.array([int(x) for x in s if x != " "])

_HEADER_LENGTH = 3
barcode_data = pd.read_csv(
    CODEBOOK_PATH, header=_HEADER_LENGTH, skipinitialspace=True,
    usecols=["name", "id", "barcode"], converters={"barcode": _parse_barcode_from_string},
)
with open(CODEBOOK_PATH) as fh:
    _header_rows = [row for i, row in enumerate(_csv.reader(fh, delimiter=",")) if i < _HEADER_LENGTH]
bit_names = [x.strip() for x in _header_rows[2][1:]]
print(f"Codebook: {CODEBOOK_PATH.name}, {len(bit_names)} bits, {len(barcode_data)} barcodes")

barcodes = np.stack(barcode_data["barcode"].values)
weights = sorted(set(barcodes.sum(axis=1).tolist()))
print(f"Barcode weight (on-bits per codeword): {weights}")

from scipy.spatial.distance import pdist
min_hamming = (pdist(barcodes, metric="hamming") * barcodes.shape[1]).min()
print(f"Minimum pairwise Hamming distance: {min_hamming:.0f}")

is_gene = ~barcode_data["name"].str.contains("Blank", case=False)
gene_names     = barcode_data.loc[is_gene, "name"].values
gene_barcodes  = barcodes[is_gene.values]
dead_bit_cols  = [b - 1 for b in DEAD_BITS]   # bit N (1-indexed) -> column N-1
n_dead_on_bits = gene_barcodes[:, dead_bit_cols].sum(axis=1)

gene_status_df = pd.DataFrame({
    "gene": gene_names,
    "n_dead_on_bits": n_dead_on_bits,
    "status": np.select(
        [n_dead_on_bits == 0, n_dead_on_bits == 1],
        ["unaffected", "error_corrected"],
        default="lost",
    ),
})
GENE_STATUS_PATH = SAMPLE_DIR / "analysis" / f"lt2_dead_bits_{DEAD_BITS[0]}_{DEAD_BITS[-1]}_gene_status.csv"
gene_status_df.to_csv(GENE_STATUS_PATH, index=False)

counts = gene_status_df["status"].value_counts()
n_genes = len(gene_status_df)
print(f"\nGenes in LT2v0 (excluding blanks): {n_genes}")
for status in ("unaffected", "error_corrected", "lost"):
    n = int(counts.get(status, 0))
    print(f"  {status:16s}: {n} ({n / n_genes:.1%})")

lost_genes = sorted(gene_status_df.loc[gene_status_df['status'] == 'lost', 'gene'])
print(f"\nLOST genes ({len(lost_genes)}):")
print(", ".join(lost_genes))
print(f"\nSaved full per-gene status: {GENE_STATUS_PATH}")